# Fine-tune ResNet-18 on the new fashion dataset

This notebook trains on the current 8,000-image dataset: eight classes with 1,000 images per class (Casual Shoes, Handbags, Jeans, Shirts, Sports Shoes, Tops, Tshirts, and Watches). It uses the existing project training/evaluation scripts.

Before running, put `colab_code.zip`, `training_images.zip` (containing a `training_images/` folder), and `clean_dataset.csv` in `MyDrive/fashion-product-classifier/`. In Colab choose **Runtime → Change runtime type → GPU**. GPU availability and free runtime limits vary.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil, zipfile

DRIVE_DIR = Path('/content/drive/MyDrive/fashion-product-classifier')
PROJECT_DIR = Path('/content/fashion-product-classifier')
DATA_DIR = PROJECT_DIR
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

required = ['colab_code.zip', 'training_images.zip', 'clean_dataset.csv']
missing = [name for name in required if not (DRIVE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing files in {DRIVE_DIR}: {missing}')

with zipfile.ZipFile(DRIVE_DIR / 'colab_code.zip') as archive:
    archive.extractall(PROJECT_DIR)
LOCAL_DATA_ZIP = Path('/content/training_images.zip')
shutil.copy2(DRIVE_DIR / 'training_images.zip', LOCAL_DATA_ZIP)
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    archive.extractall(DATA_DIR)
shutil.copy2(DRIVE_DIR / 'clean_dataset.csv', DATA_DIR / 'clean_dataset.csv')
IMAGE_DIR = DATA_DIR / 'training_images'
CSV_PATH = DATA_DIR / 'clean_dataset.csv'
print('Images:', len(list(IMAGE_DIR.glob('*.jpg'))))
print('CSV:', CSV_PATH)
print('Project:', PROJECT_DIR)

In [ ]:
# Colab already includes CUDA-enabled PyTorch in GPU runtimes. Install only the project utilities.
%pip -q install pandas scikit-learn matplotlib pillow

In [ ]:
import pandas as pd
import torch

df = pd.read_csv(CSV_PATH)
assert {'id', 'articleType'}.issubset(df.columns), df.columns.tolist()
assert df['id'].nunique() == len(df), 'Duplicate image IDs in CSV'
missing_images = [str(i) for i in df['id'] if not (IMAGE_DIR / f'{i}.jpg').is_file()]
assert not missing_images, f'Missing image files, examples: {missing_images[:5]}'
print(df['articleType'].value_counts().to_string())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys

batch_size = 32 if torch.cuda.is_available() else 16
cmd = [
    sys.executable, 'train.py', '--model', 'resnet18',
    '--csv', str(CSV_PATH), '--image-dir', str(IMAGE_DIR),
    '--epochs', '10', '--batch-size', str(batch_size),
    '--output-dir', str(PROJECT_DIR / 'checkpoints' / 'resnet18_8class'),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)

In [ ]:
import subprocess, sys

WEIGHTS = PROJECT_DIR / 'checkpoints' / 'resnet18_8class' / 'resnet18_best.pth'
EVAL_DIR = PROJECT_DIR / 'eval_output' / 'resnet18_8class'
subprocess.run([
    sys.executable, 'evaluate.py', '--model', 'resnet18',
    '--weights', str(WEIGHTS), '--csv', str(CSV_PATH),
    '--img_dir', str(IMAGE_DIR), '--imagenet_norm', '--out', str(EVAL_DIR),
], cwd=PROJECT_DIR, check=True)
subprocess.run([
    sys.executable, 'visualize_tsne.py', '--model', 'resnet18',
    '--weights', str(WEIGHTS), '--csv', str(CSV_PATH),
    '--img_dir', str(IMAGE_DIR), '--imagenet_norm',
    '--out', str(EVAL_DIR / 'tsne.png'),
], cwd=PROJECT_DIR, check=True)

In [ ]:
# Save the trained checkpoint and evaluation artifacts to Drive before the Colab runtime ends.
DRIVE_OUT = DRIVE_DIR / 'resnet18_results'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(WEIGHTS, DRIVE_OUT / 'resnet18_best.pth')
shutil.copytree(EVAL_DIR, DRIVE_OUT / 'eval_output', dirs_exist_ok=True)
shutil.make_archive(str(DRIVE_OUT / 'eval_output'), 'zip', EVAL_DIR)
print('Saved results to:', DRIVE_OUT)